In [ ]:
# paths
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import stats

from main_analyses.dual_pfc_funcs import getParams, load_dict

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')

# params
SAVE_FIG = False
data_path = 'preprocessed_data/'
params = getParams()
subjects, symbols, color_map = params['subjects'], params['markers'], params['color_map']

In [ ]:
# Determine whether angle between areas is due to neuron or trial count
angle_dat = load_dict(data_path + 'simdataset_varyThetaShuffle.pkl') # get angles from Fig 5

df = pd.DataFrame(columns=['SessionID', 'Subject', 'n_trials', 'n_left_chans', 'n_right_chans', 'angle_x1', 'angle_x2'])

# get trial info from each session
for sub in subjects:
    mat = data_path + 'all_data_delay_' + sub + '.mat'
    dat = sio.loadmat(mat, squeeze_me=True)['all_data']

    sess_names = [x for x,_ in dat.dtype.fields.items() if x.lower()[:2]==sub[:2]]
    for sess in sess_names:
        curr_dat = dat[sess].item()
        tmp = {'SessionID': sess,
               'Subject': sub,
               'n_trials': int(curr_dat['n_trials']),
               'n_left_chans': int(curr_dat['n_arr1']),
               'n_right_chans': int(curr_dat['n_arr2']),
               'angle_x1': angle_dat[sess]['neural_x1'], 
               'angle_x2': angle_dat[sess]['neural_x2']}
        df.loc[len(df)] = tmp
# df

In [ ]:
# plot angles vs number of trials and neurons
fig, ax = plt.subplots(3, 1, tight_layout=True)
for sub,sym in zip(subjects,symbols):
    tmp_df = df[df['Subject']==sub] # get subject specific sessions
    ydata = np.concatenate((tmp_df['angle_x1'],tmp_df['angle_x2']))

    # number of trials
    xdata = np.concatenate((tmp_df['n_trials'],tmp_df['n_trials']))
    ax[0].plot(xdata,ydata,marker=sym,ls='',color='k',markersize=5,fillstyle='none',label='Monkey {:s}'.format(sub[:2].title()))

    # number of neurons in left hem
    xdata = np.concatenate((tmp_df['n_left_chans'],tmp_df['n_left_chans']))
    ax[1].plot(xdata,ydata,marker=sym,ls='',color='k',markersize=5,fillstyle='none')

    # number of neurons in right hem
    xdata = np.concatenate((tmp_df['n_right_chans'],tmp_df['n_right_chans']))
    ax[2].plot(xdata,ydata,marker=sym,ls='',color='k',markersize=5,fillstyle='none')

for i in range(3):
    ax[i].set_yticks([0,30,60,90])
    ax[i].set_ylabel(r'$\theta_{estimated}$')
    
ax[0].legend(loc='lower right')
ax[0].set_xlabel('# of trials')
ax[1].set_xlabel('# of neurons in left hemisphere')
ax[2].set_xlabel('# of neurons in right hemisphere')

if SAVE_FIG:
    pdf = PdfPages('figs/theta_est_control.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# get 50% quantiles of x-axis values to compare values of theta in each quantile
all_angles = np.concatenate((df['angle_x1'], df['angle_x2']))

# a: number of trials
trial_counts = np.concatenate((df['n_trials'],df['n_trials'])) # for both x1 and x2 angles
trial_quant = np.median(trial_counts) # get 50%
low_trial_angles = all_angles[trial_counts <= trial_quant] # angles corresponding to sessions with trial counts below or equal to the median
high_trial_angles = all_angles[trial_counts > trial_quant] # angles corresponding to sessions with trial counts above the median

# b: number of neurons
# left hemisphere
left_chans = np.concatenate((df['n_left_chans'],df['n_left_chans'])) # for both x1 and x2 angles
left_quant = np.median(left_chans) # get 50%
low_left_angles = all_angles[left_chans <= left_quant] # angles corresponding to sessions with trial counts below or equal to the median
high_left_angles = all_angles[left_chans > left_quant] # angles corresponding to sessions with trial counts above the median

# right hemisphere
right_chans = np.concatenate((df['n_right_chans'],df['n_right_chans'])) # for both x1 and x2 angles
right_quant = np.median(right_chans) # get 50%
low_right_angles = all_angles[right_chans <= right_quant] # angles corresponding to sessions with trial counts below or equal to the median
high_right_angles = all_angles[right_chans > right_quant] # angles corresponding to sessions with trial counts above the median


In [ ]:
# pooled statistics
alpha = 0.01
_,ptrial = stats.ttest_ind(low_trial_angles,high_trial_angles,alternative='two-sided')
_,pleft = stats.ttest_ind(low_left_angles,high_left_angles,alternative='two-sided')
_,pright = stats.ttest_ind(low_right_angles,high_right_angles,alternative='two-sided')

print(r'Difference in mean of $\theta_{estimated}$ due to trial or neurons?')
print('Pooled, trial count: {}, p = {:.1E}'.format(ptrial<alpha,ptrial))
print('Pooled, left neuron count: {}, p = {:.1E}'.format(pleft<alpha,pleft))
print('Pooled, right neuron count: {}, p = {:.1E}'.format(pright<alpha,pright))

In [ ]:
# pooled statistics
alpha = 0.05
ptrial = stats.levene(low_trial_angles,high_trial_angles,center='median').pvalue
pleft = stats.levene(low_left_angles,high_left_angles,center='median').pvalue
pright = stats.levene(low_right_angles,high_right_angles,center='median').pvalue

print(r'Difference in varaince of $\theta_{estimated}$ due to trial or neurons?')
print('Pooled, trial count: {}, p = {:.1E}'.format(ptrial<alpha,ptrial))
print('Pooled, left neuron count: {}, p = {:.1E}'.format(pleft<alpha,pleft))
print('Pooled, right neuron count: {}, p = {:.1E}'.format(pright<alpha,pright))